# Tennis commentary: end-to-end evaluation

This notebook calls `sports_events.end_to_end_eval` modules. It defaults to reading the completed cached run. Set `RUN_REMOTE = True` to execute stages (Google Cloud authentication and paid API access required). Existing identical calls are cached.

**Comparison:** DL-selected frames → tuned VLM → tuned text LLM, versus Gemini 3.1 Flash-Lite receiving the complete unfiltered eight-second DL pilot clips. Scoring uses only the same 24 test clips (192 seconds, 24 reference events). Both systems are restricted to the same 192 seconds; the baseline sees complete clips while the pipeline sees selected windows. This is not a fair streaming-latency benchmark.

In [1]:
from pathlib import Path
import os, sys, json
root = Path.cwd()
if not (root / "src/sports_events").exists():
    root = root.parent
if not (root / "src/sports_events").exists():
    raise RuntimeError("Open from the repository or its notebooks folder")
os.chdir(root)
sys.path.insert(0, str(root / "src"))
from sports_events.end_to_end_eval import Config, prepare, run_vlm, run_pipeline, run_baseline, evaluate
from sports_events.io import read_json
RUN_REMOTE = False
config = Config()
run = Path(config.run)
print(json.dumps(config.__dict__, indent=2))

{
  "run": "runs/end_to_end_pilot_only_v3",
  "project": "gen-lang-client-0084130013",
  "llm": "projects/214431161041/locations/us/endpoints/8459061921969078272",
  "vlm": "projects/214431161041/locations/us/endpoints/4910225415601127424",
  "baseline": "gemini-3.1-flash-lite",
  "judge": "gemini-3.5-flash",
  "embedding": "gemini-embedding-001",
  "dataset": "data/tenniset_selector_vlm",
  "vlm_run": "runs/vlm-tuned-5554697559152263168-v1",
  "reference": "data/tenniset_commentary_append_only/ground-truth.jsonl",
  "clips": "data/tenniset_delta/clips",
  "video": "data/tenniset/videos/V010.mp4",
  "clip_plan": "data/tenniset_pilot/clips/preparation.json",
  "chunk_seconds": 300
}


## 1. Freeze sources and test scope
`prepare` checks the frozen VLM results, image hashes, commentary reference, clip membership, source video and protocol. It never passes reference labels to generation models.

In [2]:
if RUN_REMOTE:
    run_vlm(config)
protocol = prepare(config) if RUN_REMOTE else read_json(run / "protocol.json")
print(protocol["scope"])
print("Reference events:", len(protocol["reference"]))
print("Test clips:", len(protocol["clips"]))
print("Source video minutes (not sent in full):", round(protocol["video_seconds"] / 60, 2))

Both systems receive only the same 24 DL-pilot test clips (192 seconds). No other match footage is sent.
Reference events: 24
Test clips: 24
Source video minutes (not sent in full): 73.26


## 2. DL → tuned VLM → tuned commentary LLM
Reuse the verified 34 VLM outputs from the prior test. Invalid/null event objects are logged and skipped without ground-truth repair. The tuned LLM sees its own accumulated commentary and the preceding two predicted events. Each new sentence is appended; histories are never replaced by reference text.

In [3]:
pipeline = run_pipeline(config) if RUN_REMOTE else read_json(run / "pipeline.json")["segments"]
print("Commentary segments:", len(pipeline))
print(json.dumps(pipeline[:2], indent=2))
print("Skipped:", read_json(run / "pipeline.json")["skipped"])

Commentary segments: 33
[
  {
    "id": "tenniset_V010_001_000",
    "clip_id": "tenniset_V010_001",
    "seconds": 661.08,
    "center_seconds": 660.1400000000001,
    "text": "Azarenka hits a backhand groundstroke.",
    "event": {
      "name": "Azarenka",
      "action_base": "hit",
      "action_details": "backhand groundstroke",
      "consequence": null
    }
  },
  {
    "id": "tenniset_V010_001_001",
    "clip_id": "tenniset_V010_001",
    "seconds": 661.5600000000002,
    "center_seconds": 660.8,
    "text": "Azarenka hits another backhand groundstroke.",
    "event": {
      "name": "Azarenka",
      "action_base": "hit",
      "action_details": "backhand groundstroke",
      "consequence": null
    }
  }
]
Skipped: [{'id': 'tenniset_V010_012_002', 'reason': 'null or invalid event schema', 'raw': [{'name': 'Azarenka', 'action_base': 'hit', 'action_details': 'backhand', 'consequence': 'return'}]}]


## 3. Complete pilot-clip baseline
All source frames inside the 24 pilot test intervals are retained in separate eight-second MP4 clips; video is resized to 640 pixels wide and audio is removed to avoid using broadcast speech as a shortcut. Native model video sampling still applies. The same player-reference images are supplied. The baseline model receives no event labels, test intervals or ground-truth commentary. Its own earlier generated text supplies history.

In [4]:
baseline = run_baseline(config) if RUN_REMOTE else read_json(run / "baseline-test.json")["segments"]
full = read_json(run / "baseline-full.json")
print("Video chunks:", full["chunks_completed"], "/", full["chunks_total"])
print("All pilot-clip segments:", len(full["segments"]))
print("Segments inside test intervals:", len(baseline))
print(json.dumps(baseline[:2], indent=2))

Video chunks: 24 / 24
All pilot-clip segments: 185
Segments inside test intervals: 185
[
  {
    "id": "baseline_000_000",
    "seconds": 496.54,
    "text": "Serena Williams prepares to serve.",
    "chunk": 0,
    "clip_id": "tenniset_V010_000"
  },
  {
    "id": "baseline_000_001",
    "seconds": 497.54,
    "text": "Serena Williams hits a serve.",
    "chunk": 0,
    "clip_id": "tenniset_V010_000"
  }
]


## 4. Semantic metrics and blinded LLM judge
Embeddings: `gemini-embedding-001`, SEMANTIC_SIMILARITY, 768 dimensions. Best-match cosine is constrained to the same clip and ±3 seconds; it measures similarity, not factual accuracy.

Judge: independent `gemini-3.5-flash`. Candidate IDs and metadata are anonymized. Each clip is judged twice with A/B swapped. Matching must be one-to-one; full details earn 1, correct player/action with missing details earns 0.5, contradictions earn 0. Wrong named players receive no credit. Sparse-reference unsupported claims are distinguished from contradictions. Other metrics: ROUGE-L, exact repetition, fluency, coherence and order agreement.

In [5]:
results = evaluate(config) if RUN_REMOTE else read_json(run / "results.json")
print(json.dumps(results, indent=2))

{
  "metrics": {
    "pipeline": {
      "semantic_soft_recall": 0.8947320555565171,
      "semantic_soft_precision": 0.8610605213296414,
      "semantic_soft_f1": 0.8775734222252478,
      "exact_duplicate_fraction": 0.3333333333333333,
      "temporal_tolerance_seconds": 3.0,
      "reference_events": 24,
      "predicted_segments": 33,
      "mean_clip_rouge_l": 0.37844501655120266,
      "judge_weighted_event_recall": 0.21875,
      "judge_full_event_recall": 0.0,
      "judge_weighted_precision": 0.1590909090909091,
      "judge_contradiction_fraction": 0.2878787878787879,
      "judge_unsupported_fraction": 0.3939393939393939,
      "judge_mean_fluency": 3.857142857142857,
      "judge_mean_coherence": 2.9285714285714284,
      "judge_wins_labeled_clips": 8,
      "judge_labeled_passes": 14
    },
    "baseline": {
      "semantic_soft_recall": 0.9255029396263975,
      "semantic_soft_precision": 0.3500915960988227,
      "semantic_soft_f1": 0.5080153485350992,
      "exact_dupli

## 5. Inspect commentary alongside the reference
Read complete timestamped outputs in the run folder. The following is a compact, inspectable example; do not interpret an LLM judgment as a human label.

In [6]:
clip_id = protocol["reference"][0]["clip_id"]
for label, rows in [("Reference", protocol["reference"]), ("Pipeline", pipeline), ("Video baseline", baseline)]:
    print("\n" + label + " / " + clip_id)
    for row in rows:
        if row["clip_id"] == clip_id:
            print(f"{row['seconds']:.3f}s: {row['text']}")


Reference / tenniset_V010_001
660.100s: Williams hits a topspin backhand.
661.360s: A topspin forehand from Azarenka.
662.640s: Williams plays a topspin forehand.
663.880s: Azarenka hits a topspin forehand.
665.120s: A topspin backhand from Williams.
666.440s: Azarenka plays a topspin backhand.

Pipeline / tenniset_V010_001
661.080s: Azarenka hits a backhand groundstroke.
661.560s: Azarenka hits another backhand groundstroke.
663.160s: Azarenka hits a backhand groundstroke.
663.880s: Azarenka hits a forehand groundstroke in play.
666.000s: Azarenka hits a backhand groundstroke.
666.320s: Azarenka hits a backhand.

Video baseline / tenniset_V010_001
659.800s: Serena Williams hits a forehand.
660.500s: Victoria Azarenka hits a backhand.
661.200s: Serena Williams hits a forehand.
662.000s: Victoria Azarenka hits a backhand.
662.700s: Serena Williams hits a forehand.
663.500s: Victoria Azarenka hits a backhand.
664.200s: Serena Williams hits a forehand.
665.000s: Victoria Azarenka hits a 

## 6. Tokens, timings and cost estimates
Costs include reused VLM calls once for pipeline accounting; those were not called again. Both paths cover exactly 192 seconds of pilot clips. The old whole-match attempt did not produce baseline results and is not used here. Embeddings, availability probes, local compute, storage and any interrupted requests are not included in the estimates. See the pricing source and rates in the module.

In [7]:
print(json.dumps(read_json(run / "usage.json"), indent=2))

{
  "stages": {
    "pipeline_llm": {
      "calls": 33,
      "input_tokens": 11541,
      "output_tokens": 243,
      "thinking_tokens": 0,
      "seconds": 254.4529999999795,
      "estimated_usd": 0.0053620874999999995
    },
    "judge": {
      "calls": 48,
      "input_tokens": 35441,
      "output_tokens": 14458,
      "thinking_tokens": 57391,
      "seconds": 312.28000000014435,
      "estimated_usd": 0.6998025
    },
    "full_video": {
      "calls": 24,
      "input_tokens": 86984,
      "output_tokens": 4254,
      "thinking_tokens": 0,
      "seconds": 43.26699999999255,
      "estimated_usd": 0.028127
    },
    "reused_vlm": {
      "calls": 34,
      "input_tokens": 842258,
      "output_tokens": 1392,
      "thinking_tokens": 0,
      "seconds": 202.42500000016298,
      "estimated_usd": 0.350876625
    }
  },
  "note": "Includes reused VLM costs once for pipeline accounting; not recharged in this run. Embedding, preflight, storage and local compute costs excluded. B

## Limitations and next checks
- One reused development match and only 24 reference events.
- Ground-truth prose is synthetic verbalization of human event labels.
- Baseline sees complete eight-second clips; video rescaling and native sampling affect quality. Reference matching uses original annotation midpoints, while end-frame commitment stamps remain in metadata.
- Judge and embedding models are automated proxies; inspect player/shot errors manually.
- Compare matched-context, causal video chunks next; deduplicate DL windows; evaluate more untouched matches.

Implementation: `src/sports_events/end_to_end_eval.py`, `src/sports_events/commentary_metrics.py`. Tests: `tests/test_end_to_end_eval.py`.

## Completed pilot findings
The pipeline produced 33 entries versus 185 for direct video. Semantic soft F1 was 0.878 versus 0.508; similarity is not factual accuracy. Judge-weighted event recall was only 21.9% versus 22.9%, and neither system received full-detail credit. Exact repetition within clips was 33.3% versus 68.6%.

Only seven of the 24 clips contain the 24 reference events. Other claims remain unverified. Judge winner agreement after swapping presentation order was only 58.3%; human review is needed.

Estimated inference cost: $0.356 pipeline (including reused VLM calls), $0.028 direct video. This setup does not demonstrate cost savings. Priorities: improve player/action correctness, suppress overlapping duplicate windows, reduce image input costs, and test fresh, densely annotated matches.